In [3]:
"""
Project: Codveda Data Science Internship - Level 1 Task 1
Title: Production-Grade Web Scraping & Data Ingestion Pipeline
Author: Chukwuma Obinna Ulasi
Description: Scrapes multi-page quote records from quotes.toscrape.com using Python,
             handles pagination, parses HTML structures securely, and exports
             structured tabular data for downstream ML/Analytics pipelines.
"""


import requests
from bs4 import BeautifulSoup
import pandas as pd

# Define the target URL
url = "http://quotes.toscrape.com/"

# Send a polite HTTP request with a User-Agent header
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

# Check if the connection was successful
print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    print("Connection successful! Ready to parse HTML.")
else:
    print("Failed to connect.")

Status Code: 200
Connection successful! Ready to parse HTML.


In [6]:
# Parse the HTML content using BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

# Find all quote containers on the page
quote_containers = soup.find_all('div', class_='quote')

extracted_quotes = []

for container in quote_containers:
    # Extract text, author, and tags
    quote_text = container.find('span', class_='text').get_text(strip=True)
    author = container.find('small', class_='author').get_text(strip=True)
    tags = [tag.get_text(strip=True) for tag in container.find_all('a', class_='tag')]
    
    extracted_quotes.append({
        "quote": quote_text,
        "author": author,
        "tags": ", ".join(tags)
    })

print(f"Successfully extracted {len(extracted_quotes)} quotes from page 1!")
print("\nSample Quote:")
print(f"\"{extracted_quotes[0]['quote']}\" — {extracted_quotes[0]['author']}")

Successfully extracted 10 quotes from page 1!

Sample Quote:
"“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”" — Albert Einstein


In [7]:
import time

all_quotes = []
base_url = "http://quotes.toscrape.com/page/"
max_pages = 3

for page in range(1, max_pages + 1):
    target_url = f"{base_url}{page}/"
    print(f"Scraping page {page}: {target_url}")
    
    response = requests.get(target_url, headers={"User-Agent": "Mozilla/5.0"})
    
    # Check if page exists (handle end of pagination)
    if response.status_code == 404:
        print("Reached end of pagination (404). Stopping.")
        break
        
    soup = BeautifulSoup(response.text, 'html.parser')
    containers = soup.find_all('div', class_='quote')
    
    for container in containers:
        quote_text = container.find('span', class_='text').get_text(strip=True)
        author = container.find('small', class_='author').get_text(strip=True)
        tags = [tag.get_text(strip=True) for tag in container.find_all('a', class_='tag')]
        
        all_quotes.append({
            "quote": quote_text,
            "author": author,
            "tags": ", ".join(tags),
            "tag_count": len(tags)
        })
        
    # Polite delay between requests
    time.sleep(1)

# Convert to DataFrame
df_scraped = pd.DataFrame(all_quotes)

# Export to CSV and JSON
df_scraped.to_csv("codveda_level1_task1_quotes.csv", index=False)
df_scraped.to_json("codveda_level1_task1_quotes.json", orient='records', indent=4)

print(f"\nSuccessfully scraped a total of {len(df_scraped)} quotes across {max_pages} pages!")
print("Files exported: 'codveda_level1_task1_quotes.csv' and 'codveda_level1_task1_quotes.json'")
display(df_scraped.head())

Scraping page 1: http://quotes.toscrape.com/page/1/
Scraping page 2: http://quotes.toscrape.com/page/2/
Scraping page 3: http://quotes.toscrape.com/page/3/

Successfully scraped a total of 30 quotes across 3 pages!
Files exported: 'codveda_level1_task1_quotes.csv' and 'codveda_level1_task1_quotes.json'


,quote,author,tags,tag_count
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world",4
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices",2
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles",5
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor",4
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational",2
